# Building XLM-R for Token Classification

*Chapter 4 - Multilingual Named Entity Recognition · Natural Language Processing with Transformers (Tunstall, von Werra and Wolf)*

## Overview

Now that the data is ready, we need a model. For NER, the model must give a label to **every token**, and not one label for the whole sentence. A pretrained XLM-RoBERTa (XLM-R) model already turns each token into a vector that knows about the meaning and the context of the token. We only need to add a small layer on top that turns each vector into class scores.

In this notebook, we write this model as a PyTorch class. The class has two parts:

- **Body:** the pretrained XLM-R encoder. It is trained on text in about 100 languages and gives a hidden vector for every token.
- **Head:** a dropout layer and a linear layer. They are new and have random weights, so they must be trained on our NER data.

## Objective

- Write the class `XLMRobertaForTokenClassification` and understand the shape of every tensor.
- Create a configuration that stores the seven tag names.
- Load the pretrained weights into the new model.
- Run a forward pass on a test sentence and read the output.
- Add a helper function that shows one tag for each token.

## Workflow

| Step | What happens | Section |
|---|---|---|
| 1 | Import PyTorch, the Roberta building blocks and objects from the first notebook | Section 1 |
| 2 | Define the model class with its forward pass and loss | Section 2 |
| 3 | Build the config and load the weights | Section 3 |
| 4 | Test the untrained model on one sentence | Section 4 |
| 5 | Define `tag_text` to tag any sentence | Section 5 |

## Tools and inputs

- **PyTorch** (`torch`, `torch.nn`) for the layers and the loss.
- **`transformers`**: `RobertaPreTrainedModel` (handles weight loading and initialization), `RobertaModel` (the encoder), `XLMRobertaConfig` and `TokenClassifierOutput` (the standard result container).
- **Inputs from the first notebook:** `tags` and `xlmr_tokenizer`.
- **Model checkpoint:** `xlm-roberta-base`.

## Notes for the reader

- Hugging Face has a built-in class with the same name. We write our own version here to see how a task-specific head is added to an encoder.
- A GPU is not required, but it makes the later fine-tuning much faster. The code selects the device automatically.

## Section 1 - Setup and Imports

The first cell sets `USE_TF` to `"0"` (so TensorFlow is not loaded) and imports PyTorch, the configuration class, the output container and the two Roberta classes. It also imports `tags` and `xlmr_tokenizer` from `a_glance_at_dataset_and_tokenizer.ipynb`. The import runs that notebook, and `%%capture` hides the messages.

In [2]:
%%capture

import os
os.environ["USE_TF"] = "0"

import torch
import torch.nn as nn
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel

import import_ipynb
from a_glance_at_dataset_and_tokenizer import tags, xlmr_tokenizer

## Section 2 - The Token Classification Model

### 2.1 Design of the Class

| Component | Code | Job |
|---|---|---|
| Body | `RobertaModel(config, add_pooling_layer=False)` | Gives one hidden vector for every token. The pooling layer is not needed, because it is made for sentence-level tasks. |
| Dropout | `nn.Dropout(config.hidden_dropout_prob)` | Reduces over-fitting during training. |
| Head | `nn.Linear(config.hidden_size, config.num_labels)` | Converts each hidden vector to one score for each tag class. |
| Weight setup | `self.init_weights()` | Initializes the new head. The body weights are loaded later by `from_pretrained`. |

The class inherits from `RobertaPreTrainedModel`, so `from_pretrained` and `save_pretrained` work as usual. The attribute `config_class` tells the library which configuration to create.

### 2.2 Shapes in the Forward Pass

| Tensor | Shape |
|---|---|
| `input_ids` | `[batch_size, sequence_length]` |
| `outputs[0]` (encoder output) | `[batch_size, sequence_length, hidden_size]` |
| `logits` | `[batch_size, sequence_length, num_labels]` |
| `labels` | `[batch_size, sequence_length]` |

### 2.3 The Loss

When `labels` are given, the model calculates a **cross-entropy loss** over all tokens. The code flattens the batch and sequence axes first, so that every token is one training example:

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(z_{i,\,y_i})}{\sum_{c=1}^{C} \exp(z_{i,\,c})}
$$

- $N$ is the number of tokens whose label is **not** `-100`. Tokens with `-100` are ignored.
- $C$ is the number of classes (7 tags here).
- $z_{i,c}$ is the logit of token $i$ for class $c$, and $y_i$ is the true class of token $i$.

The class returns a `TokenClassifierOutput` with the `loss` (only if labels were given) and the `logits`.

In [3]:
class XLMRobertaForTokenClassification(RobertaPreTrainedModel):
    """XLM-RoBERTa encoder with a linear head for token classification.

    The model reads token ids and returns one score for each class and for
    each token. The body is `RobertaModel` without the pooling layer. A
    dropout layer and a linear layer on top of the encoder output create
    the logits.

    Args:
        config: `XLMRobertaConfig` that defines the encoder and `num_labels`.

    Attributes:
        num_labels: Number of tag classes.
        roberta: The XLM-R encoder (the body) without the pooling layer.
        dropout: Dropout layer applied to the encoder output.
        classifier: Linear layer that maps hidden states to class scores.
    """
    # Tells the Trainer not to pass extra loss arguments to forward()
    accepts_loss_kwargs = False
    # Configuration class that is used when we call from_pretrained()
    config_class = XLMRobertaConfig
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        # Load model body
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        # Set up token classification head
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        # Load and initialize weights
        self.init_weights()

    def forward(
        self, input_ids=None, attention_mask=None, 
        token_type_ids=None, labels=None, **kwargs
    ):
        """Run the encoder and the classification head.

        Args:
            input_ids: Token ids with shape `[batch_size, sequence_length]`.
            attention_mask: Mask with the same shape that marks real tokens (1)
                and padding (0).
            token_type_ids: Optional segment ids that are passed to the encoder.
            labels: Optional label ids with shape `[batch_size, sequence_length]`.
                Positions with the value -100 are ignored by the loss.
            **kwargs: Extra keyword arguments that are passed to the encoder.

        Returns:
            A `TokenClassifierOutput`. Its `logits` have the shape
            `[batch_size, sequence_length, num_labels]`. The `loss` is set only
            when `labels` is given. `hidden_states` and `attentions` come from
            the encoder and can be `None`.
        """
        # Use model bofy to get encoder representations
        outputs = self.roberta(
            input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, **kwargs
        )
        # Apply classifier to encoder representations
        output_sequence = self.dropout(outputs[0])
        logits = self.classifier(output_sequence)
        # Calculate losses
        loss = None
        if labels is not None:
            # Targets with the value -100 are ignored by CrossEntropyLoss
            loss_fct = nn.CrossEntropyLoss()
            # Flatten batch and sequence axes: [batch*seq, num_labels] vs [batch*seq]
            loss = loss_fct(
                logits.view(-1, self.num_labels), labels.view(-1)
            )
        # Return model output object
        return TokenClassifierOutput(
            loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions
        )

## Section 3 - Configuration and Loading Weights

### 3.1 Tag Mappings

The next cell builds two dictionaries from the names in `tags`. `index2tag` maps an id to a name, and `tag2index` maps a name to an id. They are the same as in the previous notebook and are saved in the model config in the next step.

In [4]:
index2tag = {idx:tag for idx, tag in enumerate(tags.names)}
tag2index = {tag:idx for idx, tag in enumerate(tags.names)}

### 3.2 The Model Configuration

`AutoConfig.from_pretrained` loads the settings of `xlm-roberta-base`. We add three values: `num_labels` (7, taken from `tags.num_classes`), `label2id` and `id2label`. With this information in the config, saved models can show real tag names, such as `B-PER`, instead of `LABEL_1`.

In [5]:
from transformers import AutoConfig

xlmr_model_name = "xlm-roberta-base"
# Store the number of labels and the label mappings in the model configuration
xlmr_config = AutoConfig.from_pretrained(xlmr_model_name, num_labels=tags.num_classes,
                                         label2id=tag2index, id2label=index2tag)

### 3.3 Loading the Pretrained Weights

The code chooses a CUDA GPU if one is available and the CPU if not. Then `from_pretrained` builds our class with the config and loads the pretrained weights into the body. The head does not exist in the checkpoint, so it keeps its random start values. Finally, `.to(device)` moves the model to the chosen device.

In [6]:
# Use the GPU when it is available, otherwise the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
xlmr_model = (XLMRobertaForTokenClassification
              .from_pretrained(xlmr_model_name, config=xlmr_config)
              .to(device))

## Section 4 - Testing the Untrained Model

### 4.1 Encoding a Sentence

We test the model with the sentence `"Jack Sparrow likes New York!"`. The table shows the 10 tokens and their ids. The special tokens `<s>` and `</s>` have the ids `0` and `2`.

In [7]:
import pandas as pd
text = "Jack Sparrow likes New York!"
xlmr_tokens = xlmr_tokenizer(text).tokens()
# encode() adds the special tokens and returns a tensor of shape [1, seq_len]
input_ids = xlmr_tokenizer.encode(text, return_tensors="pt")
pd.DataFrame([xlmr_tokens, input_ids[0].numpy()], index=["Tokens", "Input IDs"])

,0,1,2,3,4,5,6,7,8,9
Tokens,<s>,▁Jack,▁Spar,row,▁like,s,▁New,▁York,!,</s>
Input IDs,0,21763,37456,15555,1884,7,2356,5753,38,2


### 4.2 Forward Pass and Output Shape

The model returns one score for each class and for each token. The stored output confirms the shapes: the sentence has **10 tokens**, and the logits have the shape `torch.Size([1, 10, 7])`, that is, 1 sentence, 10 tokens and 7 classes. `argmax` over the last axis chooses the class with the highest score for each token.

> **Note:** The variable name `predicitons` has a spelling mistake in the original code. It is left as it is, because later cells use this name.

In [8]:
# logits have the shape [batch_size, sequence_length, num_labels]
outputs = xlmr_model(input_ids.to(device)).logits
# Choose the class with the highest score for every token
predicitons = torch.argmax(outputs, dim=-1)
print(f"Number of tokens in sequence: {len(xlmr_tokens)}")
print(f"Shape of outputs: {outputs.shape}")

Number of tokens in sequence: 10
Shape of outputs: torch.Size([1, 10, 7])


### 4.3 Reading the Predictions

The stored table shows the same tag, `I-PER`, for **all** 10 tokens. This is expected. The body knows the language, but the head has random weights and has never seen an NER label. The result shows that fine-tuning on labeled data is necessary.

In [9]:
# Convert the predicted class ids to tag names
preds = [tags.names[p] for p in predicitons[0].cpu().numpy()]
pd.DataFrame([xlmr_tokens, preds], index=['Tokens', 'Tags'])

,0,1,2,3,4,5,6,7,8,9
Tokens,<s>,▁Jack,▁Spar,row,▁like,s,▁New,▁York,!,</s>
Tags,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER


## Section 5 - A Helper for Tagging Text

The function `tag_text` repeats the steps of Section 4 for any sentence: tokenize, run the model, take the `argmax` and return a table of tokens and tags. We will use it after fine-tuning to check the model on German and French sentences. This cell only defines the function, so there is no stored output.

In [11]:
def tag_text(text, tags, model, tokenizer):
    """Predict the tag of every token in a text.

    Args:
        text: Input sentence as a plain string.
        tags: `ClassLabel` feature that maps class ids to tag names.
        model: Token classification model that returns logits.
        tokenizer: Tokenizer that matches the model.

    Returns:
        A `pandas.DataFrame` with the rows `Tokens` and `Tags` and one column
        for each token (special tokens included).

    Notes:
        The input ids are moved to the global `device`, so the model must be
        on the same device.
    """
    tokens = tokenizer(text).tokens()
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    outputs = model(input_ids).logits
    # The argmax over the class axis (dim=2) gives one tag id for every token
    predictions = torch.argmax(outputs, dim=2)
    preds = [tags.names[p] for p in predictions[0].cpu().numpy()]
    return pd.DataFrame([tokens, preds], index=["Tokens", "Tags"])

## Conclusion

We built a token classification model from an XLM-R encoder and a new linear head, loaded the pretrained weights and tested the forward pass.

| Item | Value in this notebook |
|---|---|
| Checkpoint | `xlm-roberta-base` |
| Number of classes | 7 (`O`, `B-PER`, `I-PER`, `B-ORG`, `I-ORG`, `B-LOC`, `I-LOC`) |
| Test sentence | `"Jack Sparrow likes New York!"` (10 tokens) |
| Logits shape | `[1, 10, 7]` |
| Prediction before training | `I-PER` for all 10 tokens |

The last row is the main lesson: the head must be trained before the model can be used.

### What comes next

`performance_measures.ipynb` explains how to measure NER quality with entity-level metrics and defines `align_predictions`. After that, `fine-tuning-xlm-roberta.ipynb` trains this model.

## **Resources & References**

**Official Documentation**

- [Hugging Face Transformers: Token classification guide](https://huggingface.co/docs/transformers/tasks/token_classification)
- [Hugging Face Transformers: `Trainer` and `TrainingArguments`](https://huggingface.co/docs/transformers/main_classes/trainer)
- [Model card: `FacebookAI/xlm-roberta-base`](https://huggingface.co/FacebookAI/xlm-roberta-base)
- [PyTorch: `CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

**Key Papers / Research**

- Conneau, A. et al. (2019). [Unsupervised Cross-lingual Representation Learning at Scale](https://arxiv.org/abs/1911.02116).
- Devlin, J., Chang, M.-W., Lee, K., Toutanova, K. (2018). [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805).

**Additional Reading**

- Tunstall, L., von Werra, L., Wolf, T. (2022). [Natural Language Processing with Transformers, Revised Edition](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/). O'Reilly Media. Chapter 4: Multilingual Named Entity Recognition.